# Ingest API

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook ingests data via API. **Default: Incremental Load** (only new data is ingested). Set the load type widget to 'full' for a full reload.

---

**Parameters:**
- `api_url`: Source API endpoint
- `output_table`: Destination table name
- `load_type`: 'incremental' (default) or 'full'

---

**Instructions:**
1. Set the API URL, output table, and load type using the widgets below.
2. Run all cells to ingest data as per the selected load type.


In [ ]:
# Databricks widgets for parameterization
import requests
import pyspark.sql.functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ingest_api")

dbutils.widgets.text("api_url", "", "API URL")
dbutils.widgets.text("output_table", "bronze_table", "Output Table")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"], "Load Type")
api_url = dbutils.widgets.get("api_url")
output_table = dbutils.widgets.get("output_table")
load_type = dbutils.widgets.get("load_type")

try:
    response = requests.get(api_url)
    response.raise_for_status()
    data = response.json()
    df = spark.createDataFrame(data)
    logger.info(f"Loaded {df.count()} records from {api_url}")
    if load_type == "full":
        df.write.format("delta").mode("overwrite").saveAsTable(output_table)
        logger.info(f"Full load: Overwrote {output_table}")
    else:
        df.write.format("delta").mode("append").saveAsTable(output_table)
        logger.info(f"Incremental load: Appended to {output_table}")
    print(f"Ingestion ({load_type}) successful.")
except Exception as e:
    logger.error(f"Error in ingestion: {e}")
    print(f"Error: {e}")


## API URL and Header Helpers
Functions to resolve API URLs and build request headers.

In [ ]:
def _resolve_api_url(api_profile: dict, source: dict) -> str:
    endpoint = source.get('api_endpoint', '')
    base_url = api_profile.get('base_url', '')
    if not endpoint:
        raise ValueError('api_endpoint is required in source metadata')
    if endpoint.startswith('http://') or endpoint.startswith('https://'):
        return endpoint
    if not base_url:
        raise ValueError('base_url is required in API profile for relative api_endpoint values')
    return f'{base_url.rstrip('/')}/{endpoint.lstrip('/')}'

def _resolve_headers(api_profile: dict, headers: dict | None = None) -> dict:
    final_headers = dict(headers or {})
    auth_type = str(api_profile.get('auth_type', 'none')).lower()
    if auth_type == 'bearer':
        token_env = api_profile.get('token_env', '')
        token = os.getenv(token_env, '') if token_env else ''
        if token:
            final_headers.setdefault('Authorization', f'Bearer {token}')
    elif auth_type == 'api_key':
        key_env = api_profile.get('api_key_env', '')
        key_header = api_profile.get('api_key_header', 'X-API-Key')
        api_key = os.getenv(key_env, '') if key_env else ''
        if api_key:
            final_headers.setdefault(key_header, api_key)
    elif auth_type == 'basic':
        import base64
        user_env = api_profile.get('user_env', '')
        password_env = api_profile.get('password_env', '')
        user = os.getenv(user_env, '') if user_env else ''
        password = os.getenv(password_env, '') if password_env else ''
        if user and password:
            token = base64.b64encode(f'{user}:{password}'.encode()).decode()
            final_headers.setdefault('Authorization', f'Basic {token}')
    return final_headers

## API Response Extraction
Function to extract records from API response payloads.

In [ ]:
def _extract_records(payload: object, source: dict) -> list[dict]:
    # ...existing code...

---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure data was ingested. Add more tests as needed for your data.


In [ ]:
# Simple validation
try:
    df_test = spark.table(output_table)
    assert df_test.count() > 0, "No records found in output table."
    print("Validation passed: Data ingested.")
except Exception as e:
    print(f"Validation failed: {e}")


---

## Resource Cleanup

Unpersist DataFrames and clean up resources if needed to avoid memory issues in production workflows.

In [ ]:
# Resource cleanup
if 'df_test' in locals():
    df_test.unpersist(blocking=True)
    print("Resources cleaned up.")
